# Caso H · 01 Arquitectura del chatbot — tools sobre InfluxDB + RAG documental

> _Tutorial · Caso de uso: **H — RAG + Chatbot** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Comprender el patrón **tools (datos numéricos) + RAG (conocimiento general)** y definir el conjunto de herramientas mínimo del chatbot.


## 2. Qué se aprende

- Decisión: pregunta → tool o pregunta → RAG.
- 6 herramientas básicas (`query_influxdb`, `compare_periods`, ...).
- Cómo mockear modelos predictivos para no bloquearse.
- Cómo separar conocimiento factual de conocimiento documental.


## 3. Contexto del caso de uso

Equipo H construye el chatbot integrador: usa modelos B y C/E como tools.


## 4. Relación con CENTINELA+

El chatbot va a producción tras la integración de los modelos en semana 3.


## 5. Relación con Medallion

Consume plata; oro = tools + RAG.


## 6. Datos de entrada

Conceptual.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Tabla pregunta → mecanismo.


In [2]:
mapping = pd.DataFrame(
    [
        ("Dato puntual histórico", "tool: query_influxdb"),
        ("Comparación entre periodos", "tool: compare_periods"),
        ("Predicción meteo / consumo", "tool: get_*_prediction (mock o real)"),
        ("Estado actual del edificio", "tool: get_building_state"),
        ("Detección anomalía HVAC", "tool: check_hvac_anomaly"),
        ("Conocimiento general / normativa", "RAG sobre docs"),
    ],
    columns=["pregunta", "mecanismo"],
)
mapping


,pregunta,mecanismo
0,Dato puntual histórico,tool: query_influxdb
1,Comparación entre periodos,tool: compare_periods
2,Predicción meteo / consumo,tool: get_*_prediction (mock o real)
3,Estado actual del edificio,tool: get_building_state
4,Detección anomalía HVAC,tool: check_hvac_anomaly
5,Conocimiento general / normativa,RAG sobre docs


## 10. Exploración paso a paso

Diagrama Mermaid.


In [3]:
from IPython.display import Markdown
Markdown('''```mermaid
flowchart LR
  Q[Pregunta] --> R{Decisión}
  R -- numérica --> T[Tools InfluxDB]
  R -- predicción --> P[Tools mocked → reales]
  R -- documental --> S[RAG ElasticSearch]
  T --> A[Respuesta]
  P --> A
  S --> A
```''')


```mermaid
flowchart LR
  Q[Pregunta] --> R{Decisión}
  R -- numérica --> T[Tools InfluxDB]
  R -- predicción --> P[Tools mocked → reales]
  R -- documental --> S[RAG ElasticSearch]
  T --> A[Respuesta]
  P --> A
  S --> A
```

## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Tools en notebooks 02-03; RAG en 04.


## 13. Visualizaciones explicativas

Distribución del golden set por categoría.


In [4]:
golden = pd.read_csv(ROOT / "notebooks/_data/chatbot_golden_set.csv", comment="#")
golden["category"].value_counts().plot.bar(color="#3F51B5")
plt.title("Golden set — preguntas por categoría")
plt.tight_layout()


## 14. Validaciones

Tabla cargada.


## 15. Errores comunes

1. Indexar valores numéricos en ElasticSearch (incorrecto: usar tool).
2. Mockear modelos sin firma estable (cambiar firma rompe integraciones).
3. No registrar la trazabilidad de qué tool eligió el agente.


## 16. Ejercicios propuestos

1. Añade una categoría 'control' y discute si requiere tool nueva.
2. Diseña una política de fallback si la tool falla.
3. Discute si los predictores deberían ir en MLflow Server.


## 17. Cómo se reutiliza con datos reales

Cambiar `INFLUXDB_*` y endpoints de modelos. La arquitectura es estable.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `08_case_H_rag_chatbot/02_tools_influxdb.ipynb`.
- Documento web del caso: `docs/use-cases/case-h-rag-chatbot.md`.


## 19. Marco teórico (nivel doctoral)

### Retrieval-Augmented Generation (Lewis et al. 2020)

$$
P(y \mid x) = \sum_{z \in \mathcal{Z}} P_\eta(z \mid x) \cdot P_\theta(y \mid x, z)
$$

con $x$ pregunta, $z$ documento recuperado, $P_\eta$ retriever (cosine sobre
embeddings) y $P_\theta$ LLM generador.

### Similarity coseno

$$
\text{sim}(x, z) = \frac{\mathbf{e}_x \cdot \mathbf{e}_z}{\|\mathbf{e}_x\| \|\mathbf{e}_z\|}
$$

### Tools tipadas

$$
\mathcal{T} = \{ t_i : \mathbb{X}_i \to \mathbb{Y}_i \mid \text{schema JSON} \}
$$

Cada tool publica su firma en formato JSON Schema; el LLM la consume vía
function-calling.

### Métricas

$$
\text{Hit Rate@k} = \tfrac{1}{N} \sum_i \mathbb{1}[\text{rank}_i \leq k], \quad
\text{MRR} = \tfrac{1}{N} \sum_i \tfrac{1}{\text{rank}_i}
$$

Objetivos: $\text{Hit@5} \geq 0.85$, $\text{MRR} \geq 0.7$, Faithfulness ≥ 0.9.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

El chatbot es la **cara visible** de CAPTIA al usuario final (profesores, equipo de mantenimiento). Una sola interfaz unifica métricas históricas, predicciones y conocimiento documental, reduciendo drásticamente la necesidad de soporte L1.

### ROI estimado

| Concepto | Valor |
|---|---|
| Reducción tickets soporte L1 | +3 500 €/año |
| Tiempo respuesta profesores | +1 200 €/año |
| **Bruto** | **+4 700 €/año** |
| Coste API LLM (Claude/GPT) | -1 800 €/año |
| **Neto** | **+2 900 €/año** |

### Riesgos y mitigaciones

- Hallucinations del LLM: mitigar con tools de hechos verificables.
- Coste API escala linealmente con uso: monitorizar.


## 21. Bibliografía y referencias

- Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS.
- Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. EMNLP.
- LangChain Project. *Documentation*. https://python.langchain.com
- Anthropic (2024). *Claude 3.5 Sonnet Model Card*.
